<a href="https://colab.research.google.com/github/miriamamin1213-ux/Seed42_Models/blob/main/GPT242_TabularHancock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# GPT2_42_HANCOCK
# Same GPT2 architecture as GPT2_42_Hecktor
# No SMOTE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score
)

from transformers import GPT2Model

# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):

    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


SEED = 42
seed_everything(SEED)

# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

# ============================================================
# Read Data
# ============================================================

with open(
    "/content/drive/MyDrive/clinical_data.json"
) as f:
    clinical = pd.DataFrame(
        json.load(f)
    )

with open(
    "/content/drive/MyDrive/pathological_data.json"
) as f:
    pathology = pd.DataFrame(
        json.load(f)
    )

df = clinical.merge(
    pathology,
    on="patient_id",
    how="inner"
)

df = df[
    df["hpv_association_p16"].isin(
        ["positive", "negative"]
    )
].copy()

df["HPV"] = df[
    "hpv_association_p16"
].map({
    "negative": 0,
    "positive": 1
})

# ============================================================
# Build Modelling Dataset
# ============================================================

df_model = df[
[
    "age_at_initial_diagnosis",
    "sex",
    "smoking_status",
    "primarily_metastasis",
    "first_treatment_intent",
    "first_treatment_modality",
    "days_to_first_treatment",
    "adjuvant_treatment_intent",
    "adjuvant_radiotherapy",
    "adjuvant_radiotherapy_modality",
    "adjuvant_systemic_therapy",
    "adjuvant_systemic_therapy_modality",
    "adjuvant_radiochemotherapy",
    "primary_tumor_site",
    "pT_stage",
    "pN_stage",
    "histologic_type",
    "number_of_positive_lymph_nodes",
    "number_of_resected_lymph_nodes",
    "perinodal_invasion",
    "lymphovascular_invasion_L",
    "vascular_invasion_V",
    "perineural_invasion_Pn",
    "resection_status",
    "infiltration_depth_in_mm",
    "HPV"
]
].copy()

# ============================================================
# Missing Values
# ============================================================

cat_cols = [
    "sex",
    "smoking_status",
    "primarily_metastasis",
    "first_treatment_intent",
    "first_treatment_modality",
    "adjuvant_treatment_intent",
    "adjuvant_radiotherapy",
    "adjuvant_radiotherapy_modality",
    "adjuvant_systemic_therapy",
    "adjuvant_systemic_therapy_modality",
    "adjuvant_radiochemotherapy",
    "primary_tumor_site",
    "pT_stage",
    "pN_stage",
    "histologic_type",
    "perinodal_invasion",
    "lymphovascular_invasion_L",
    "vascular_invasion_V",
    "perineural_invasion_Pn",
    "resection_status"
]

for col in cat_cols:

    df_model[col] = df_model[col].fillna(
        df_model[col].mode()[0]
    )

num_cols = [
    "age_at_initial_diagnosis",
    "days_to_first_treatment",
    "number_of_positive_lymph_nodes",
    "number_of_resected_lymph_nodes",
    "infiltration_depth_in_mm"
]

for col in num_cols:

    df_model[col] = df_model[col].fillna(
        df_model[col].median()
    )

# ============================================================
# Label Encoding
# ============================================================

for col in cat_cols:

    df_model[col] = LabelEncoder().fit_transform(
        df_model[col].astype(str)
    )

# ============================================================
# Features / Target
# ============================================================

X = df_model.drop(
    columns=["HPV"]
)

y = df_model["HPV"]

feature_columns = X.columns.tolist()

print("Number of features:", len(feature_columns))

# ============================================================
# Train Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("\nTrain samples:")
print(y_train.value_counts())

print("\nTest samples:")
print(y_test.value_counts())

# ============================================================
# Standardisation
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================================
# SMOTE
# ============================================================

from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=SEED
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())

# ============================================================
# Torch Conversion
# ============================================================

X_train = torch.FloatTensor(
    X_train_smote
)

X_test = torch.FloatTensor(
    X_test
)

y_train = torch.LongTensor(
    y_train_smote
)

y_test = torch.LongTensor(
    y_test.values
)

# ============================================================
# DataLoaders
# ============================================================

BATCH_SIZE = 256

train_dataset = TensorDataset(
    X_train,
    y_train
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# ============================================================
# GPT2 Model
# ============================================================

class HPVNetGPT2Numerical(nn.Module):

    def __init__(
        self,
        num_features=25,
        num_classes=2,
        model_name="openai-community/gpt2",
        dropout=0.1,
        freeze_gpt2=True
    ):

        super().__init__()

        self.num_features = num_features
        self.freeze_gpt2 = freeze_gpt2

        self.gpt2 = GPT2Model.from_pretrained(
            model_name
        )

        self.gpt2.config.use_cache = False

        hidden_size = (
            self.gpt2.config.hidden_size
        )

        self.feature_projection = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.GELU(),
            nn.LayerNorm(hidden_size)
        )

        self.feature_embedding = nn.Parameter(
            torch.empty(
                1,
                num_features,
                hidden_size
            )
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

        nn.init.normal_(
            self.feature_embedding,
            mean=0.0,
            std=0.02
        )

        if freeze_gpt2:
            self.gpt2.requires_grad_(False)

    def train(self, mode=True):

        super().train(mode)

        if self.freeze_gpt2:
            self.gpt2.eval()

        return self

    def forward(self, x):

        x = x.unsqueeze(-1)

        x = self.feature_projection(x)

        x = x + self.feature_embedding

        outputs = self.gpt2(
            inputs_embeds=x,
            use_cache=False,
            return_dict=True
        )

        hidden_states = (
            outputs.last_hidden_state
        )

        pooled_output = hidden_states.mean(
            dim=1
        )

        logits = self.classifier(
            pooled_output
        )

        return logits

# ============================================================
# Model Setup
# ============================================================

model = HPVNetGPT2Numerical(
    num_features=len(feature_columns),
    num_classes=2,
    freeze_gpt2=True
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

# ============================================================
# Training
# ============================================================

EPOCHS = 100

best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss_sum = 0.0
    train_count = 0

    for batch_x, batch_y in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = model(batch_x)

        train_loss = criterion(
            outputs,
            batch_y
        )

        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(
                lambda p: p.requires_grad,
                model.parameters()
            ),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum += (
            train_loss.item()
            *
            batch_x.size(0)
        )

        train_count += batch_x.size(0)

    avg_train_loss = (
        train_loss_sum / train_count
    )

    model.eval()

    test_loss_sum = 0.0
    test_count = 0

    with torch.no_grad():

        for batch_x, batch_y in test_loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)

            test_loss = criterion(
                outputs,
                batch_y
            )

            test_loss_sum += (
                test_loss.item()
                *
                batch_x.size(0)
            )

            test_count += batch_x.size(0)

    avg_test_loss = (
        test_loss_sum / test_count
    )

    if avg_test_loss < best_val_loss:

        best_val_loss = avg_test_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            "best_gpt2_hancock.pth"
        )

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch+1}, "
            f"Train={avg_train_loss:.4f}, "
            f"Test={avg_test_loss:.4f}"
        )

# ============================================================
# Load Best Model
# ============================================================

model.load_state_dict(
    torch.load(
        "best_gpt2_hancock.pth",
        map_location=device
    )
)

model.eval()

all_probs = []
all_preds = []
all_true = []

with torch.no_grad():

    for batch_x, batch_y in test_loader:

        batch_x = batch_x.to(device)

        outputs = model(batch_x)

        probs = torch.softmax(
            outputs,
            dim=1
        )

        preds = torch.argmax(
            probs,
            dim=1
        )

        all_probs.append(
            probs.cpu()
        )

        all_preds.append(
            preds.cpu()
        )

        all_true.append(
            batch_y.cpu()
        )

probabilities = torch.cat(
    all_probs
).numpy()

predicted = torch.cat(
    all_preds
).numpy()

y_true = torch.cat(
    all_true
).numpy()

# ============================================================
# Evaluation
# ============================================================

print()
print("Classification Report\n")

print(
    classification_report(
        y_true,
        predicted,
        digits=4,
        zero_division=0
    )
)

cm = confusion_matrix(
    y_true,
    predicted
)

print("Confusion Matrix")
print(cm)

accuracy = (
    (predicted == y_true).sum()
    /
    len(y_true)
)

bal_acc = balanced_accuracy_score(
    y_true,
    predicted
)

f1 = f1_score(
    y_true,
    predicted,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    probabilities[:,1]
)

print()
print(f"Accuracy:            {accuracy:.4f}")
print(f"Balanced Accuracy:   {bal_acc:.4f}")
print(f"F1-score:            {f1:.4f}")
print(f"AUC:                 {auc:.4f}")

# ============================================================
# SUMMARY
# ============================================================

print()
print("====================================")
print("GPT2_42_HANCOCK SUMMARY")
print("====================================")
print("Model : GPT2 Tabular")
print("Input Features :", len(feature_columns))
print("Learning Rate : 0.001")
print("Optimiser : AdamW")
print("Class Weights : None")
print("SMOTE : Yes")
print("Epochs :", EPOCHS)
print("Best Epoch :", best_epoch)
print("Best Test Loss :", round(best_val_loss,4))
print("Training Patients :", len(y_train))
print("Test Patients :", len(y_test))
print("Seed :", SEED)

Mounted at /content/drive
Using device: cuda
Number of features: 25

Train samples:
HPV
0    152
1    113
Name: count, dtype: int64

Test samples:
HPV
0    39
1    28
Name: count, dtype: int64

After SMOTE:
HPV
0    152
1    152
Name: count, dtype: int64


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 10, Train=0.6949, Test=0.6790
Epoch 20, Train=0.6786, Test=0.6919
Epoch 30, Train=0.6738, Test=0.7232
Epoch 40, Train=0.6442, Test=0.6298
Epoch 50, Train=0.6031, Test=0.5712
Epoch 60, Train=0.5559, Test=0.4932
Epoch 70, Train=0.5003, Test=0.4482
Epoch 80, Train=0.4941, Test=0.4411
Epoch 90, Train=0.4451, Test=0.4221
Epoch 100, Train=0.4711, Test=0.4318

Classification Report

              precision    recall  f1-score   support

           0     0.8462    0.8462    0.8462        39
           1     0.7857    0.7857    0.7857        28

    accuracy                         0.8209        67
   macro avg     0.8159    0.8159    0.8159        67
weighted avg     0.8209    0.8209    0.8209        67

Confusion Matrix
[[33  6]
 [ 6 22]]

Accuracy:            0.8209
Balanced Accuracy:   0.8159
F1-score:            0.7857
AUC:                 0.8892

GPT2_42_HANCOCK SUMMARY
Model : GPT2 Tabular
Input Features : 25
Learning Rate : 0.001
Optimiser : AdamW
Class Weights : None
SMOTE : Yes
